In [ ]:
# ML Return Prediction for Financial Time Series
# Luciano Prado
#
# Objective:
# Build a simple machine learning pipeline using linear regression
# to predict next-day returns from historical market features.
#
# This notebook covers:
# 1. Data download
# 2. Feature engineering
# 3. Linear regression model
# 4. Out-of-sample evaluation
# 5. Simple trading backtest


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
ticker = "SPY"
start_date = "2018-01-01"
end_date = "2024-12-31"

df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
df = df.reset_index()

print(df.head())
print(df.shape)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["Date"], df["Adj Close"])
plt.title(f"{ticker} Adjusted Close Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

In [ ]:
data = df.copy()
data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date").reset_index(drop=True)

# Daily returns
data["return"] = data["Adj Close"].pct_change()

# Lagged returns
data["lag_1"] = data["return"].shift(1)
data["lag_2"] = data["return"].shift(2)
data["lag_3"] = data["return"].shift(3)

# Rolling volatility
data["vol_5"] = data["return"].rolling(5).std()
data["vol_20"] = data["return"].rolling(20).std()

# Moving-average signal
data["ma_5"] = data["Adj Close"].rolling(5).mean()
data["ma_20"] = data["Adj Close"].rolling(20).mean()
data["ma_ratio"] = data["ma_5"] / data["ma_20"]

# Momentum
data["momentum_5"] = data["Adj Close"] / data["Adj Close"].shift(5) - 1

# Volume feature
data["volume_change"] = data["Volume"].pct_change()

# Target: next-day return
data["target"] = data["return"].shift(-1)

data = data.dropna().reset_index(drop=True)

feature_cols = [
    "lag_1",
    "lag_2",
    "lag_3",
    "vol_5",
    "vol_20",
    "ma_ratio",
    "momentum_5",
    "volume_change",
]

print(data[["Date"] + feature_cols + ["target"]].head())
print(data.shape)

In [ ]:
train_size = 0.8
split_idx = int(len(data) * train_size)

train_df = data.iloc[:split_idx].copy()
test_df = data.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_test = test_df[feature_cols]
y_test = test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

train_df["prediction"] = model.predict(X_train)
test_df["prediction"] = model.predict(X_test)

print("Model trained successfully.")

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

train_df["prediction"] = model.predict(X_train)
test_df["prediction"] = model.predict(X_test)

print("Model trained successfully.")

In [ ]:
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print(coef_df)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, test_df["prediction"], alpha=0.5)
plt.title("Predicted vs Actual Returns")
plt.xlabel("Actual Return")
plt.ylabel("Predicted Return")
plt.tight_layout()
plt.show()

In [ ]:
test_df["position"] = (test_df["prediction"] > 0).astype(int)
test_df["strategy_return"] = test_df["position"].shift(1).fillna(0) * test_df["return"]

test_df["buy_hold_curve"] = (1 + test_df["return"]).cumprod()
test_df["strategy_curve"] = (1 + test_df["strategy_return"]).cumprod()

print(test_df[["Date", "return", "prediction", "position", "strategy_return"]].head())

In [ ]:
test_df["position"] = (test_df["prediction"] > 0).astype(int)
test_df["strategy_return"] = test_df["position"].shift(1).fillna(0) * test_df["return"]

test_df["buy_hold_curve"] = (1 + test_df["return"]).cumprod()
test_df["strategy_curve"] = (1 + test_df["strategy_return"]).cumprod()

print(test_df[["Date", "return", "prediction", "position", "strategy_return"]].head())

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(test_df["Date"], test_df["buy_hold_curve"], label="Buy & Hold")
plt.plot(test_df["Date"], test_df["strategy_curve"], label="ML Strategy")
plt.title(f"Equity Curve - {ticker}")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Interpretation:")
print("- This model predicts next-day returns using linear regression.")
print("- Performance should be interpreted cautiously because financial returns are noisy.")
print("- The purpose of this notebook is to demonstrate a clean ML workflow for finance.")
print("- In real applications, transaction costs, slippage, and walk-forward validation should be added.")